In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

if not os.getcwd().endswith("/quotaclimat"):
    os.chdir("../../../../..")

repo_root_path = os.path.abspath(os.path.dirname(os.getcwd()))
if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)
repo_root_path


'/root/Workspace'

In [2]:
from quotaclimat.data_ingestion.advertising.s01_detection.e02_create_chunks import (
    ChunkCreatorJob,
    debug_split,
)
from quotaclimat.data_ingestion.advertising.s01_detection.processor import (
    chunk_creator,
)
from quotaclimat.data_ingestion.advertising.tools.segments import Segment
from quotaclimat.data_ingestion.advertising.tools.mediatree.bucket_mediatree import (
    _get_s3_file_basename,
)
from quotaclimat.data_ingestion.advertising.s01_detection.tools.visualizer.split_visualizer import (
    generate_split_visualizer,
)

from datetime import timedelta, datetime


In [3]:
def get_job_from_start(
    channel: str,
    start_sec: float,
    has_previous_segment: bool = False,
    with_next_segment: bool = False,
) -> ChunkCreatorJob:
    """
    Builds a ChunkCreatorJob for the 2-minutes mediatree audio part covering
    `start_sec` (epoch), pointing at the already-downloaded local mp3 (see
    e00_download_audio / notebooks/e01_download_media.ipynb to fetch it first).

    Set `has_previous_segment=True` to mimic a job that is not the first of a
    run (peaks in the first `seconds_reserved_for_previous_segment` seconds are
    dropped). Set `with_next_segment=True` to append the following part's
    margin, exactly like the real pipeline does for non-last jobs.
    """
    start_date = datetime.fromtimestamp(start_sec)
    # floored to the 2min interval mediatree parts are stored at
    rounded_start_date = start_date - timedelta(
        minutes=start_date.minute % 2, seconds=start_date.second, microseconds=start_date.microsecond
    )
    rounded_end_date = rounded_start_date + timedelta(minutes=2)

    segment = Segment(start_date=rounded_start_date, end_date=rounded_end_date, channel=channel)

    audio_file = _get_s3_file_basename(channel, rounded_start_date, rounded_end_date) + ".mp3"
    audio_file_path = "./.cache/mediatree/" + audio_file

    next_audio_file_path = None
    if with_next_segment:
        next_start = rounded_end_date
        next_end = next_start + timedelta(minutes=2)
        next_file = _get_s3_file_basename(channel, next_start, next_end) + ".mp3"
        next_audio_file_path = "./.cache/mediatree/" + next_file

    return ChunkCreatorJob(
        segment=segment,
        audio_file_path=audio_file_path,
        has_previous_segment=has_previous_segment,
        next_audio_file_path=next_audio_file_path,
    )


In [4]:
# Pick two audio windows to compare — e.g. one that splits correctly and one
# you suspect is mis-split (missed cut, or an over-eager cut in the middle of speech).
channel = "tf1"

focus_epoch_a = 1787809779.58
focus_epoch_b = 1787597266.18


job_a = get_job_from_start(channel, focus_epoch_a)
job_b = get_job_from_start(channel, focus_epoch_b)

trace_a = debug_split(job_a, chunk_creator)


DEBUG: audio window splitting analysis
  segment: [2026-08-27 05:48:00 -> 2026-08-27 05:50:00]  channel=tf1
  audio_file_path: ./.cache/mediatree/tf1_2026-08-27T05-48-00Z_2026-08-27T05-50-00Z.mp3
  has_previous_segment=False  next_audio_file_path=None

[1] Load audio: 5760000 samples @ 48000Hz = 120.00s
    total window duration (with margin): 120.00s, 11251 frames

[2] Silence mask (local percentile threshold)
    silence_percentile=8.0  window=±2s  smoothing=0.1s  margin=0.1
    silent frames: 1670/11251 (14.8%)

[3] Peak candidates (deepest point of each silence region)
    57 silence regions found -> 57 candidate peaks
      region[228:256] -> t=2.62s energy=0.00409
      region[289:321] -> t=3.26s energy=0.00362
      region[490:518] -> t=5.35s energy=0.00775
      region[567:594] -> t=6.24s energy=0.00623
      region[676:701] -> t=7.39s energy=0.00491
      region[1041:1089] -> t=11.52s energy=0.00000
      region[1408:1449] -> t=15.21s energy=0.00593
      region[1644:1664] -> 

In [5]:
trace_b = debug_split(job_b, chunk_creator)


DEBUG: audio window splitting analysis
  segment: [2026-08-24 18:46:00 -> 2026-08-24 18:48:00]  channel=tf1
  audio_file_path: ./.cache/mediatree/tf1_2026-08-24T18-46-00Z_2026-08-24T18-48-00Z.mp3
  has_previous_segment=False  next_audio_file_path=None

[1] Load audio: 5760000 samples @ 48000Hz = 120.00s
    total window duration (with margin): 120.00s, 11251 frames

[2] Silence mask (local percentile threshold)
    silence_percentile=8.0  window=±2s  smoothing=0.1s  margin=0.1
    silent frames: 1868/11251 (16.6%)

[3] Peak candidates (deepest point of each silence region)
    64 silence regions found -> 64 candidate peaks
      region[83:134] -> t=1.30s energy=0.00646
      region[411:459] -> t=4.79s energy=0.00000
      region[757:786] -> t=8.21s energy=0.01282
      region[1082:1103] -> t=11.67s energy=0.00454
      region[1111:1160] -> t=12.25s energy=0.00811
      region[1850:1898] -> t=19.86s energy=0.00000
      region[2216:2239] -> t=23.78s energy=0.00506
      region[2265:2285

In [6]:
html = generate_split_visualizer(
    job_a=job_a,
    job_b=job_b,
    chunk_creator=chunk_creator,
    label_a=f"Window A ({job_a.segment.start_date})",
    label_b=f"Window B ({job_b.segment.start_date})",
    focus_epoch_a=focus_epoch_a,  # zoom in on the exact timestamp you're investigating
    focus_epoch_b=focus_epoch_b,
    zoom_sec=10.0,  # +/- seconds of context around the focus timestamp
)

with open(".cache/split_comparison.html", "w") as f:
    f.write(html)


In [7]:
test_cases = [
    ("tf1", 1787809779.58, 1787597266.18, "astronautes"),
    ("tf1", 1788012620.16, 1787655199.68, "promo koh lanta"),
    ("tf1", 1787334761.54, 1787204794.18, "mat mut"),
]

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
dir_name = f".cache/split_comparison/{ts}"
os.makedirs(dir_name)

for channel, focus_epoch_a, focus_epoch_b, name in test_cases:
    html = generate_split_visualizer(
        job_a=get_job_from_start(channel, focus_epoch_a),
        job_b=get_job_from_start(channel, focus_epoch_b),
        chunk_creator=chunk_creator,
        label_a="Window A",
        label_b="Window B",
        focus_epoch_a=focus_epoch_a,  # zoom in on the exact timestamp you're investigating
        focus_epoch_b=focus_epoch_b,
        zoom_sec=10.0,  # +/- seconds of context around the focus timestamp
    )

    with open(f"{dir_name}/{name}.html", "w") as f:
        f.write(html)